# 注意力机制

In [1]:
import torch
import torch.nn as nn
import math

## 1.注意力公式
注意力计算公式：
$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

假设 Batch=B, Seq_len=S, Num_Heads=H, Head_Dim=D
1. 线性投影后：Q 形状为 [B, S, H * D]
2. 切分多头后：转置为 [B, H, S, D]
3. 注意力分数计算：Q @ K^T -> [B, H, S, D] @ [B, H, D, S] -> [B, H, S, S]
4. 乘以 Value：Scores @ V -> [B, H, S, S] @ [B, H, S, D] -> [B, H, S, D]
5. 最后合并多头：转置回 [B, S, H, D] 并 view 成 [B, S, H * D]。

## 2.KV Cache
KV Cache是指在Transformer模型中缓存键（Key）和值（Value）向量的机制。它的主要作用是在生成序列时，避免重复计算已经计算过的键和值，从而提高推理速度和效率。KV Cache通常用于自回归生成任务，如文本生成和语言建模。

KV Cache的工作原理如下：
1. **缓存键和值**：在每一层的注意力机制中，计算得到的键和值向量会被存储在缓存中，以便在后续的时间步中重复使用。
2. **避免重复计算**：在生成新的序列时，模型可以直接从缓存中获取之前计算的键和值，而不需要重新计算，从而节省计算资源。
3. **提高推理速度**：通过使用KV Cache，模型在生成长序列时可以显著减少计算量，提高推理速度，尤其是在处理大规模语言模型时。

KV Cache是Attention机制中优化的重要部分，读取巨量的KV Cache 会面临严重的显存容量瓶颈和内存带宽瓶颈 (Memory-bound)，导致推理极慢。

## 3.MHA（Multi-Head Attention）
标准的MHA中，每个注意力头都有独立的Q、K、V。如果有h个注意力头，那就有h组K和h组V需要缓存。
- h个Q头，h个K头，h个V头
- 每组Q/K/V独立，互不共享

## 4.MQA（Multi-Query Attention）
所有Q头共享同一组K和V。也就是说，只有1个K头和1个V头，但仍然有h个Q 头。
-  h个Q头，1个K头，1个V 头
-  所有Q头共享同一份K和V

## 5. GQA（Grouped-Query Attention）
GQA是MHA和MQA的折中方案。将h个Q头分成g个组（每组h/g个Q头），每组共享一组K和V。
- h个Q头，g个K头，g个V头
- 每h/g个Q头共享一组K和V
- 当g=h时退化为MHA，当g=1时退化为 MQA

In [2]:
def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
    """
    将 KV 头复制 n_rep 次，以匹配 Query 头的数量 (GQA/MQA 需要)
    """
    batch, num_kv_heads, slen, head_dim = hidden_states.shape
    if n_rep == 1:
        return hidden_states
    hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_kv_heads, n_rep, slen, head_dim)
    return hidden_states.reshape(batch, num_kv_heads * n_rep, slen, head_dim)

class GroupedQueryAttention(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int, num_kv_heads: int = None):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads if num_kv_heads is not None else num_heads
        
        self.num_queries_per_kv = self.num_heads // self.num_kv_heads
        self.head_dim = hidden_dim // num_heads
        
        # 定义投影矩阵
        self.q_proj = nn.Linear(hidden_dim, num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(hidden_dim, self.num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(hidden_dim, self.num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(num_heads * self.head_dim, hidden_dim, bias=False)

    def forward(
        self, 
        x: torch.Tensor, 
        attention_mask: torch.Tensor = None, 
        kv_cache: tuple[torch.Tensor, torch.Tensor] = None
    ):
        batch_size, seq_len, _ = x.shape
        
        # 1. 线性投影
        xq, xk, xv = self.q_proj(x), self.k_proj(x), self.v_proj(x)
        
        # ==========================================
        # TODO 1: Reshape xq, xk, xv 以适配多头注意力计算
        # 提示: 先把最后一维映射成 [num_heads, head_dim] / [num_kv_heads, head_dim]，再把头维挪到前面
        # xq = ???
        # xk = ???
        # xv = ???
        # ==========================================
        xq = xq.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        xk = xk.view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
        xv = xv.view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)

        # ==========================================
        # TODO 2: 处理 KV Cache
        # 提示: 如果有 cache，就把历史 KV 接到当前 KV 前面
        # ==========================================
        if kv_cache is not None:
            k_cache, v_cache = kv_cache
            # xk = ???
            # xv = ???
            xk = torch.cat([k_cache, xk], dim=2)
            xv = torch.cat([v_cache, xv], dim=2)
            
        new_kv_cache = (xk, xv)
        
        # 通过 repeat_kv 把 GQA 的 KV 头数扩充到和 Query 数量一致
        xk = repeat_kv(xk, self.num_queries_per_kv)
        xv = repeat_kv(xv, self.num_queries_per_kv)
        
        # ==========================================
        # TODO 3: 计算注意力分数 (Scaled Dot-Product)
        # ==========================================
        # scores = ???
        scores = torch.matmul(xq, xk.transpose(2, 3)) / math.sqrt(self.head_dim)
        
        if attention_mask is not None:
            scores = scores + attention_mask
            
        # probs = ???
        # output = ???
        probs = torch.nn.functional.softmax(scores, dim=-1)
        output = torch.matmul(probs, xv)
        
        # ==========================================
        # TODO 4: 恢复形状并输出
        # [B, H, S, D] -> [B, S, H*D]
        # ==========================================
        # output = ???
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        
        return self.o_proj(output), new_kv_cache

In [3]:
# 运行此单元格以测试你的实现
def test_mha_mqa_gqa():
    try:
        batch_size, seq_len, hidden_dim, num_heads = 2, 16, 128, 4
        
        # 1. 测试 MHA
        print("Testing MHA (Multi-Head Attention)...")
        mha = GroupedQueryAttention(hidden_dim, num_heads, num_kv_heads=num_heads)
        x = torch.randn(batch_size, seq_len, hidden_dim)
        out, _ = mha(x)
        assert out.shape == (batch_size, seq_len, hidden_dim), "MHA 输出形状错误!"
        
        # 2. 测试 GQA
        print("Testing GQA (Grouped-Query Attention)...")
        gqa = GroupedQueryAttention(hidden_dim, num_heads, num_kv_heads=2)
        out, _ = gqa(x)
        assert out.shape == (batch_size, seq_len, hidden_dim), "GQA 输出形状错误!"
        
        # 3. 测试 KV Cache
        print("Testing KV Cache Autoregressive Decoding...")
        prefill_len = 5
        x_prefill = torch.randn(batch_size, prefill_len, hidden_dim)
        _, kv_cache = mha(x_prefill)
        
        x_decode = torch.randn(batch_size, 1, hidden_dim)
        out_decode, new_kv_cache = mha(x_decode, kv_cache=kv_cache)
        assert new_kv_cache[0].shape == (batch_size, num_heads, prefill_len + 1, hidden_dim // num_heads), "KV Cache 更新错误!"
        
        print("\n✅ All Tests Passed! Attention 算子实现通过测试。")
    except NotImplementedError:
        print("请先完成 TODO 部分的代码！")
        raise
    except (AttributeError, NameError, TypeError, ValueError) as e:
        if isinstance(e, AttributeError):
            print("代码未完成，无法找到必要的属性")
        elif isinstance(e, NameError):
            print("代码可能未完成，导致了变量未定义")
        elif isinstance(e, TypeError):
            print("代码可能未完成，导致了类型错误")
        else:
            print("代码可能未完成，导致了张量维度错误")
        raise NotImplementedError("请先完成 TODO 部分的代码！") from e
    except Exception as e:
        print(f"\n❌ 测试失败，请检查张量维度: {e}")
        raise

test_mha_mqa_gqa()

Testing MHA (Multi-Head Attention)...
Testing GQA (Grouped-Query Attention)...
Testing KV Cache Autoregressive Decoding...

✅ All Tests Passed! Attention 算子实现通过测试。
